# 04 — Gap Analysis: Who Fills the Gap?

This is the core analytical notebook. We quantify:

1. **The gap** — How much less is the US contributing to humanitarian aid and ODA in 2025
   relative to its 2018–2024 baseline?
2. **The response** — Are other donors increasing their contributions? By how much?
   Is there evidence they are deliberately compensating for the US shortfall?
3. **The deficit** — How much of the gap remains unfilled? Which countries and sectors
   are most exposed to the remaining deficit?

## Analytical framework

We use **two independent data sources** that measure different things:

| Dataset | What it measures | Time lag | Strength |
|---------|-----------------|----------|----------|
| OECD DAC CRS | Broad ODA — loans, grants, technical assistance | 1–2 years | Complete, standardised |
| OCHA FTS | Humanitarian-only aid flows | Weeks–months | Timely, granular |

For the gap analysis we primarily use **FTS** (more timely, covers 2025) and
cross-validate with **OECD** (more complete, but only through 2024).

## US baseline

We define the US baseline as the **2022–2024 average** — the three most recent full
years before the January 2025 executive order freeze. Using a 3-year average smooths
out year-to-year volatility (Ukraine emergency spikes, COVID recovery, etc.) while
remaining recent enough to reflect the current aid architecture.

**Input:** `data/processed/combined.csv` from `03_clean_data.ipynb`

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
PROC_DIR = PROJECT_ROOT / "data" / "processed"
CHART_DIR = PROJECT_ROOT / "output" / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

# ── Load cleaned data ─────────────────────────────────────────────────────────
combined = pd.read_csv(PROC_DIR / "combined.csv", low_memory=False)
oecd = pd.read_csv(PROC_DIR / "oecd_clean.csv", low_memory=False)
fts = pd.read_csv(PROC_DIR / "fts_clean.csv", low_memory=False)

print(f"Combined: {len(combined):,} rows")
print(f"OECD:     {len(oecd):,} rows")
print(f"FTS:      {len(fts):,} rows")
print(f"\nCombined years: {sorted(combined['year'].unique())}")
print(f"Sources:        {combined['source'].unique()}")

## Part A — The Gap: Measuring the US shortfall

### A1. FTS-based gap (humanitarian aid, near real-time)

FTS covers 2018–2025 and is the most current signal we have. We compute:
- US baseline = average annual humanitarian contribution 2022–2024
- US 2025 (annualised) = 2025 YTD × (12 / months elapsed)
- Gap = baseline − 2025 annualised

**Annualisation caveat:** Reporting lag means 2025 YTD is an undercount.
The annualised gap is therefore a *minimum estimate*.

In [ ]:
import datetime

BASELINE_YEARS = [2022, 2023, 2024]
TODAY = datetime.date.today()
MONTHS_ELAPSED_2025 = TODAY.month + (TODAY.day / 30)  # fractional months

fts_us = fts[fts["donor_canonical"] == "United States"].copy()

# Baseline: US average annual humanitarian funding 2022–2024
us_baseline_by_yr = (
    fts_us[fts_us["year"].isin(BASELINE_YEARS)]
    .groupby("year")["amount_2023usd"]
    .sum()
)
us_baseline_mean = us_baseline_by_yr.mean()

# 2025 YTD and annualised estimate
us_2025_ytd = fts_us[fts_us["year"] == 2025]["amount_2023usd"].sum()
us_2025_annualised = us_2025_ytd / MONTHS_ELAPSED_2025 * 12

# Gap
gap_fts = us_baseline_mean - us_2025_annualised
gap_pct = gap_fts / us_baseline_mean * 100

print("=== FTS-based US humanitarian aid gap (constant 2023 USD) ===")
print(f"\nUS annual humanitarian aid — baseline period ({BASELINE_YEARS[0]}–{BASELINE_YEARS[-1]}):")
for yr, val in us_baseline_by_yr.items():
    print(f"  {yr}: ${val/1e9:.3f}B")
print(f"  Average: ${us_baseline_mean/1e9:.3f}B")

print(f"\n2025 YTD ({TODAY.strftime('%B %d')}): ${us_2025_ytd/1e9:.3f}B")
print(f"  Months elapsed: {MONTHS_ELAPSED_2025:.1f}")
print(f"  Annualised (÷{MONTHS_ELAPSED_2025:.1f} × 12): ${us_2025_annualised/1e9:.3f}B")

print(f"\nEstimated annual gap: ${gap_fts/1e9:.3f}B ({gap_pct:.1f}% below baseline)")
print(f"⚠ This is a minimum estimate — 2025 reporting is incomplete.")

### A2. OECD-based gap (broad ODA, 2018–2024)

OECD covers all ODA — not just humanitarian — and is the authoritative measure for
total US development assistance. 2024 is the last available year. We use 2024 vs
the 2022–2024 average to see if there was already a pre-2025 decline in US ODA.

In [ ]:
oecd_us = oecd[oecd["donor_canonical"] == "United States"].copy()
oecd_all = oecd.copy()

# US total ODA by year (disbursements, constant 2023 USD)
us_oda_by_yr = (
    oecd_us.groupby("year")["disbursement_2023usd"].sum() / 1e9
).rename("US ODA ($B 2023)")

all_oda_by_yr = (
    oecd_all.groupby("year")["disbursement_2023usd"].sum() / 1e9
).rename("Total ODA ($B 2023)")

oda_summary = pd.concat([us_oda_by_yr, all_oda_by_yr], axis=1)
oda_summary["US share (%)"] = (oda_summary["US ODA ($B 2023)"] / oda_summary["Total ODA ($B 2023)"] * 100).round(1)
oda_summary["US YoY change"] = us_oda_by_yr.pct_change().mul(100).round(1)

print("US ODA trajectory (OECD DAC, constant 2023 USD):")
print(oda_summary.to_string())

us_baseline_oecd = us_oda_by_yr[BASELINE_YEARS].mean()
us_2024_oecd = us_oda_by_yr.get(2024, float("nan"))
print(f"\nBaseline ({BASELINE_YEARS[0]}–{BASELINE_YEARS[-1]} avg): ${us_baseline_oecd:.2f}B")
if not np.isnan(us_2024_oecd):
    print(f"2024 actual:                   ${us_2024_oecd:.2f}B")
    print(f"Pre-2025 drift:                {(us_2024_oecd - us_baseline_oecd)/us_baseline_oecd*100:+.1f}%")

## Part B — The Response: Are other donors stepping up?

### B1. Aggregate non-US donor growth

The simplest version of the question: did total non-US humanitarian funding increase
in 2025 relative to 2024? We use FTS because it covers 2025; OECD cannot answer this.

In [ ]:
# Non-US donors only
fts_non_us = fts[fts["donor_canonical"] != "United States"].copy()

# For 2025, YTD only — compare equivalent window in 2024
# We use reported flows regardless of date for annual totals (since not all flows have dates),
# and date-filtered for the YTD comparison.

non_us_by_yr = fts_non_us.groupby("year")["amount_2023usd"].sum() / 1e9

# Annualise 2025 non-US the same way we annualised US
non_us_2025_ytd = non_us_by_yr.get(2025, 0)
non_us_2025_ann = non_us_2025_ytd / MONTHS_ELAPSED_2025 * 12
non_us_baseline = non_us_by_yr[BASELINE_YEARS].mean()

print("Non-US donor humanitarian aid by year (constant 2023 USD):")
for yr, val in non_us_by_yr.items():
    delta = f" ({(val - non_us_by_yr.get(yr-1,val))/non_us_by_yr.get(yr-1,val)*100:+.1f}% vs prev yr)" if yr > non_us_by_yr.index.min() else ""
    print(f"  {yr}: ${val:.3f}B{delta}")

print(f"\n  Baseline ({BASELINE_YEARS[0]}–{BASELINE_YEARS[-1]} avg): ${non_us_baseline:.3f}B")
print(f"  2025 annualised:              ${non_us_2025_ann:.3f}B")
gap_filled_pct = (non_us_2025_ann - non_us_baseline) / gap_fts * 100 if gap_fts > 0 else float("nan")
print(f"  Response vs baseline:         {(non_us_2025_ann - non_us_baseline)/non_us_baseline*100:+.1f}%")
if not np.isnan(gap_filled_pct):
    print(f"  Equivalent to {gap_filled_pct:.1f}% of estimated US gap")

### B2. Individual donor responses

Which specific donors grew most in 2025? We look at donors with ≥\$50M baseline
and rank by absolute increase in 2025 vs their 2022–2024 average. This tells us
**who is actually replacing US funding** — not just who is growing from a small base.

In [ ]:
MIN_BASELINE = 50e6  # $50M minimum baseline contribution

# Annual totals per non-US donor across all years
donor_annual = (
    fts_non_us.groupby(["donor_canonical", "year"])["amount_2023usd"]
    .sum()
    .unstack(fill_value=0)
)

# Compute baseline average for each donor
available_baseline_yrs = [y for y in BASELINE_YEARS if y in donor_annual.columns]
donor_annual["baseline_avg"] = donor_annual[available_baseline_yrs].mean(axis=1)

# Filter: only donors with ≥$50M baseline
donor_significant = donor_annual[donor_annual["baseline_avg"] >= MIN_BASELINE].copy()

# Annualised 2025 for each donor
if 2025 in donor_annual.columns:
    donor_significant["annualised_2025"] = donor_annual[2025] / MONTHS_ELAPSED_2025 * 12
    donor_significant["absolute_increase"] = donor_significant["annualised_2025"] - donor_significant["baseline_avg"]
    donor_significant["pct_change"] = (donor_significant["absolute_increase"] / donor_significant["baseline_avg"] * 100).round(1)

    # Rank by absolute dollar increase
    top_responders = donor_significant.sort_values("absolute_increase", ascending=False).head(15)

    print("Donors increasing most in 2025 (vs 2022–2024 baseline, annualised, constant 2023 USD):")
    print(f"{'Donor':<40} {'Baseline ($M)':>14} {'2025 ann. ($M)':>14} {'Change ($M)':>12} {'Change %':>10}")
    print("-" * 94)
    for donor, row in top_responders.iterrows():
        direction = "▲" if row["absolute_increase"] > 0 else "▼"
        print(f"{donor:<40} {row['baseline_avg']/1e6:>14,.0f} {row['annualised_2025']/1e6:>14,.0f} "
              f"{row['absolute_increase']/1e6:>+12,.0f} {direction}{row['pct_change']:>9.1f}%")
else:
    print("2025 FTS data not yet available in combined dataset.")

### B3. Counterfactual — what would 'full gap-fill' require?

We calculate how much each major donor would need to increase to collectively cover
the entire US shortfall. This is a journalistic tool — it contextualises the scale of
the gap relative to what's plausible from existing donor budgets.

In [ ]:
# The unfilled gap so far
actual_increase = donor_significant["absolute_increase"].clip(lower=0).sum() if 2025 in donor_annual.columns else 0
remaining_gap = max(0, gap_fts - actual_increase)

print(f"GAP ACCOUNTING (constant 2023 USD):")
print(f"  Estimated US annual shortfall:  ${gap_fts/1e9:.3f}B")
print(f"  Increase from other donors:     ${actual_increase/1e9:.3f}B (annualised)")
print(f"  Remaining unfilled gap:         ${remaining_gap/1e9:.3f}B")
print(f"  Gap coverage rate:              {actual_increase/gap_fts*100:.1f}%" if gap_fts > 0 else "")
print()

# What % increase would the EU / Germany / UK each need to fully fill the gap?
# Use their 2022–2024 baseline as the starting point.
CANDIDATE_DONORS = ["European Union", "Germany", "United Kingdom", "Japan", "France", "Saudi Arabia"]

print("How much would each major donor need to increase to fill the ENTIRE remaining gap?")
print(f"(Remaining gap: ${remaining_gap/1e9:.2f}B)\n")
print(f"{'Donor':<30} {'Baseline ($B)':>14} {'Required increase':>18} {'Required % raise':>17}")
print("-" * 82)

for candidate in CANDIDATE_DONORS:
    row = donor_significant.get(candidate) if candidate in donor_significant.index else None
    if row is None and candidate in donor_annual.index:
        baseline = donor_annual.loc[candidate, "baseline_avg"]
    elif row is not None:
        baseline = row["baseline_avg"]
    else:
        baseline = 0

    if baseline > 0:
        required_pct = remaining_gap / baseline * 100
        print(f"{candidate:<30} {baseline/1e9:>14.3f} {remaining_gap/1e9:>17.3f}B {required_pct:>16.0f}%")
    else:
        print(f"{candidate:<30} {'(no data)':>14}")

## Part C — The Deficit: Who is left behind?

Even if some donors partially fill the aggregate gap, the replacement funding may
not flow to the same countries or sectors. This section identifies the **structural
deficit** — countries and crises where the US shortfall is least likely to be
offset by other donors based on their existing funding patterns.

In [ ]:
# Per-country: US share of total humanitarian inflows in 2024 baseline
# (using FTS since it has country-level destination data)

fts_2024 = fts[fts["year"] == 2024].copy()

country_total_2024 = (
    fts_2024[fts_2024["recipient_country"].notna()]
    .groupby("recipient_country")["amount_2023usd"]
    .sum()
)
country_us_2024 = (
    fts_2024[(fts_2024["donor_canonical"] == "United States") & fts_2024["recipient_country"].notna()]
    .groupby("recipient_country")["amount_2023usd"]
    .sum()
)

country_exposure = pd.DataFrame({
    "total_2024": country_total_2024,
    "us_2024": country_us_2024,
}).fillna(0)
country_exposure["us_pct"] = (country_exposure["us_2024"] / country_exposure["total_2024"] * 100).round(1)
country_exposure["exposure_usd"] = country_exposure["us_2024"]  # dollar value at risk

# Filter: only countries with ≥$5M total humanitarian funding
significant_recipients = country_exposure[country_exposure["total_2024"] >= 5e6].sort_values("us_pct", ascending=False)

print("Countries most exposed to US withdrawal (2024 baseline, FTS):")
print(f"{'Country':<35} {'Total ($M)':>11} {'US ($M)':>11} {'US share':>10} {'Exposure'}")
print("-" * 80)
for country, row in significant_recipients.head(20).iterrows():
    if row["us_pct"] >= 50:
        risk = "🔴 EXTREME"
    elif row["us_pct"] >= 30:
        risk = "🟠 HIGH"
    elif row["us_pct"] >= 15:
        risk = "🟡 MODERATE"
    else:
        risk = "🟢 LOW"
    print(f"{country:<35} {row['total_2024']/1e6:>11,.0f} {row['us_2024']/1e6:>11,.0f} {row['us_pct']:>9.1f}%  {risk}")

In [ ]:
# Are other donors funding the same high-exposure countries?
# We look at 2025 FTS flows to the top 10 high-exposure countries from non-US donors.

top_exposed = significant_recipients.head(10).index.tolist()
fts_2025 = fts[fts["year"] == 2025]

print("Non-US donor funding to high-exposure countries: 2024 vs 2025 YTD\n")
print(f"{'Country':<35} {'Non-US 2024 ($M)':>17} {'Non-US 2025 YTD ($M)':>20} {'Trend':>8}")
print("-" * 84)

for country in top_exposed:
    nonUS_2024 = fts_2024[
        (fts_2024["recipient_country"] == country) &
        (fts_2024["donor_canonical"] != "United States")
    ]["amount_2023usd"].sum()

    nonUS_2025 = fts_2025[
        (fts_2025["recipient_country"] == country) &
        (fts_2025["donor_canonical"] != "United States")
    ]["amount_2023usd"].sum()

    # Annualise 2025 for comparability
    nonUS_2025_ann = nonUS_2025 / MONTHS_ELAPSED_2025 * 12

    if nonUS_2024 > 0:
        trend_pct = (nonUS_2025_ann - nonUS_2024) / nonUS_2024 * 100
        trend_str = f"{trend_pct:+.1f}%"
    else:
        trend_str = "N/A"

    print(f"{country:<35} {nonUS_2024/1e6:>17,.0f} {nonUS_2025/1e6:>20,.0f} {trend_str:>8}")

print("\n(2025 YTD is not annualised in the column above — compare cautiously)")

## Part D — Summary gap table

This consolidated table is the analytical output that feeds the article. It shows,
for each major crisis country, the estimated US funding shortfall and the degree
to which other donors are compensating.

In [ ]:
# Build the master gap table
rows = []
for country in significant_recipients.head(15).index:
    total_24 = country_exposure.loc[country, "total_2024"] if country in country_exposure.index else 0
    us_24 = country_exposure.loc[country, "us_2024"] if country in country_exposure.index else 0
    us_pct_24 = country_exposure.loc[country, "us_pct"] if country in country_exposure.index else 0

    # 2025 non-US funding (YTD × annualisation)
    nonUS_25_ytd = fts_2025[
        (fts_2025["recipient_country"] == country) &
        (fts_2025["donor_canonical"] != "United States")
    ]["amount_2023usd"].sum()
    nonUS_24 = fts_2024[
        (fts_2024["recipient_country"] == country) &
        (fts_2024["donor_canonical"] != "United States")
    ]["amount_2023usd"].sum()

    nonUS_25_ann = nonUS_25_ytd / MONTHS_ELAPSED_2025 * 12
    other_increase = nonUS_25_ann - nonUS_24
    us_shortfall = us_24  # the amount the US contributed in 2024 that may not come in 2025
    net_deficit = us_shortfall - max(0, other_increase)

    rows.append({
        "country": country,
        "total_2024_usd": total_24,
        "us_2024_usd": us_24,
        "us_dependency_pct": us_pct_24,
        "other_increase_ann": other_increase,
        "estimated_net_deficit": net_deficit,
        "gap_covered_pct": max(0, other_increase) / us_24 * 100 if us_24 > 0 else 0,
    })

gap_table = pd.DataFrame(rows).set_index("country")

# Save this table for use in visualise.py and the article
gap_table.to_csv(PROC_DIR / "gap_table.csv")

print("=== MASTER GAP TABLE (constant 2023 USD) ===")
print(f"  Annualised based on {MONTHS_ELAPSED_2025:.1f} months of 2025 data\n")
print(f"{'Country':<35} {'US 2024':>10} {'US dep%':>8} {'Others↑':>10} {'Net deficit':>12} {'Covered%':>10}")
print("-" * 88)
for country, row in gap_table.iterrows():
    print(
        f"{country:<35} "
        f"${row['us_2024_usd']/1e6:>8,.0f}M "
        f"{row['us_dependency_pct']:>7.1f}% "
        f"${row['other_increase_ann']/1e6:>+8,.0f}M "
        f"${row['estimated_net_deficit']/1e6:>10,.0f}M "
        f"{row['gap_covered_pct']:>9.1f}%"
    )

print(f"\nSaved → {PROC_DIR / 'gap_table.csv'}")

In [ ]:
# ── Chart: bubble chart — US dependency vs gap coverage ───────────────────────
# X-axis: US dependency (% of 2024 humanitarian funding from US)
# Y-axis: Gap coverage by others in 2025 (%)
# Bubble size: Total 2024 funding (absolute scale of crisis)
# Quadrants:
#   Bottom-right (high dependency, low coverage) = most at risk
#   Top-right (high dependency, high coverage)   = others stepped up
#   Bottom-left (low dependency, low coverage)   = less exposed
#   Top-left (low dependency, high coverage)     = already well-covered

plot_data = gap_table[gap_table["us_2024_usd"] >= 5e6].copy()

fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(
    plot_data["us_dependency_pct"],
    plot_data["gap_covered_pct"].clip(0, 150),
    s=plot_data["total_2024_usd"] / 1e5,  # scale bubble by funding
    c=plot_data["estimated_net_deficit"],
    cmap="RdYlGn_r",
    alpha=0.75,
    edgecolors="white",
    linewidths=0.8,
)

plt.colorbar(scatter, ax=ax, label="Estimated net deficit ($)")

# Label each country
for country, row in plot_data.iterrows():
    ax.annotate(
        country,
        (row["us_dependency_pct"], row["gap_covered_pct"].clip(0, 150)),
        fontsize=7.5, ha="left", va="bottom",
        xytext=(3, 3), textcoords="offset points",
    )

# Quadrant lines
ax.axvline(30, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axhline(50, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)

# Quadrant labels
ax.text(5, 140, "Low exposure", fontsize=8, color="gray", style="italic")
ax.text(65, 140, "Exposed but covered", fontsize=8, color="steelblue", style="italic")
ax.text(65, 5, "HIGH RISK: gap not filled", fontsize=8.5, color="red", fontweight="bold")
ax.text(5, 5, "Low exposure", fontsize=8, color="gray", style="italic")

ax.set_xlabel("US share of 2024 humanitarian funding (%)", fontsize=11)
ax.set_ylabel("Gap covered by other donors in 2025 YTD (%)", fontsize=11)
ax.set_title(
    "US Aid Dependency vs. Gap Coverage by Other Donors\n"
    "Bubble size = total 2024 humanitarian inflows",
    fontsize=13, fontweight="bold"
)
ax.set_xlim(0, 100)
ax.set_ylim(0, 155)

plt.tight_layout()
chart_path = CHART_DIR / "gap_bubble_chart.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {chart_path}")

In [ ]:
# ── Final narrative summary ────────────────────────────────────────────────────
print("=" * 65)
print("GAP ANALYSIS — KEY NUMBERS FOR THE ARTICLE")
print("=" * 65)

print(f"""
The Gap (FTS-based, humanitarian aid)
───────────────────────────────────────
  US 2022–2024 baseline:     ${us_baseline_mean/1e9:.2f}B / year
  US 2025 annualised:        ${us_2025_annualised/1e9:.2f}B / year (⚠ undercount)
  Estimated annual gap:      ${gap_fts/1e9:.2f}B ({gap_pct:.0f}% below baseline)

The Response
─────────────
  Non-US donor baseline:     ${non_us_baseline:.2f}B / year
  Non-US 2025 annualised:    ${non_us_2025_ann:.2f}B / year
  Increase vs baseline:      ${(non_us_2025_ann - non_us_baseline)/1e9:+.2f}B
  Coverage of US gap:        {gap_filled_pct:.0f}%
  Remaining unfilled gap:    ${remaining_gap/1e9:.2f}B

Countries most at risk
───────────────────────
  Extreme dependence (>50% US share): """, end="")
extreme = significant_recipients[significant_recipients["us_pct"] >= 50].index.tolist()
print(", ".join(extreme[:6]) if extreme else "(none above 50%)")
print(f"""  High dependence (30–50%):          """, end="")
high = significant_recipients[
    (significant_recipients["us_pct"] >= 30) &
    (significant_recipients["us_pct"] < 50)
].index.tolist()
print(", ".join(high[:6]) if high else "(none in range)")

print(f"""
Caveats
────────
  • 2025 annualised values based on {MONTHS_ELAPSED_2025:.1f} months of data
  • Reporting lag means 2025 figures will rise — all gaps are minimum estimates
  • FTS is self-reported; not all donors report consistently
  • OECD data only available through 2024; 2025 ODA data won't appear until late 2026
""")
print("=" * 65)